# Merging Authors and Books 

In [132]:
import pandas as pd
from sqlalchemy import create_engine, text
import configparser

# Leer el archivo de configuración
config = configparser.ConfigParser()
config.read('../goodreads_config.cfg')

# Obtener las credenciales de la base de datos
db_user = config['database']['user']
db_password = config['database']['password']
db_host = config['database']['host']
db_name = config['database']['database']

# Crear conexión a la base de datos
engine = create_engine(f'mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}')

# Leer la tabla goodreads_authors desde la base de datos
goodreads_authors_df = pd.read_sql('SELECT * FROM goodreads_authors', engine)

# Leer el fichero CSV
goodreads_library_export_df = pd.read_csv('../data/goodreads_library_export.csv')

# Hacer un merge de ambos dataframes basado en la columna 'name' de goodreads_authors y 'Author' del CSV
merged_df = pd.merge(goodreads_authors_df, goodreads_library_export_df, left_on='name', right_on='Author', how='inner')


In [133]:
# Elimina la columna 'Author' del dataframe
merged_df.drop(columns=['Author'], inplace=True)

# Actualiza los encabezados de las columnas a minúsculas y con guiones bajos
merged_df.columns = merged_df.columns.str.lower().str.replace(' ', '_')

In [134]:
# Create a new column 'age' with the age of the author died - born
# For example: 
#                        name        born        died                  age
#0          Madeleine L'Engle  1918-11-29  2007-09-06  32423 days, 0:00:00
#1               Harold Bloom  1930-07-11  2019-10-14  32602 days, 0:00:00
# Put age in years
merged_df['age'] = (pd.to_datetime(merged_df['died']) - pd.to_datetime(merged_df['born'])).dt.days / 365.25

In [135]:
merged_df.shape

(1455, 44)

In [136]:
# Coloca la columna 'age' en la 4 posiición desde la izquierda
merged_df = merged_df[['author_id', 'name', 'born', 'age', 'died'] + [col for col in merged_df.columns if col not in ['author_id', 'name', 'born', 'age', 'died']]]

# coloca la columna 'author_id' en primer lugar a la izquierda
merged_df = merged_df[['author_id', 'name', 'born', 'died'] + [col for col in merged_df.columns if col not in ['author_id', 'name', 'born', 'died']]]

# Elimina rows duplicadas basadas en la columna 'author_id', quedándote con aquella aparición en la que el campo 'my_review' no sea nulo ni esté vacío
merged_df = merged_df.drop_duplicates(subset='author_id', keep='first')

In [137]:
merged_df['isbn'] = merged_df['isbn'].str.replace('=', '').str.replace('"', '')
merged_df['isbn13'] = merged_df['isbn13'].str.replace('=', '').str.replace('"', '')

In [138]:
# 5️⃣ Subir el DataFrame como una tabla temporal en MySQL
merged_df.to_sql('temp_goodreads_authors_library', engine, index=False, if_exists='replace')

# 6️⃣ Crear la vista (`VIEW`) en MySQL Workbench
with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS view_goodreads_authors_library"))  # Borrar vista si existe
    conn.execute(text("""
        CREATE VIEW view_goodreads_authors_library AS 
        SELECT * FROM temp_goodreads_authors_library
    """))  # Crear la nueva vista
    conn.commit()

